# Blender 4.2 Cycles GPU — 1-Click 3D Short Prototype (Willebrord Snellius)

Works on **Free Google Colab (T4 GPU)** & **Free Kaggle Notebooks (T4 x2 / P100 GPU)**.
1. Select **Runtime -> Change runtime type -> T4 GPU** (Colab) or **Accelerator -> GPU T4** (Kaggle).
2. Click **Run All** — 10x faster with **OPTIX Tensor-Core Denoiser + Persistent GPU VRAM**!

In [ ]:
# 1. Verify Free NVIDIA GPU & Clone Repo
!nvidia-smi
!rm -rf snellius-blender-gpu-prototype
!git clone https://github.com/ssghtc002-sudo/snellius-blender-gpu-prototype.git
%cd snellius-blender-gpu-prototype

In [ ]:
# 2. Download & Extract Blender 4.2.3 LTS Headless (~25 sec)
import os
if not os.path.exists('/tmp/blender/blender'):
    !wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /tmp/blender.tar.xz
    !mkdir -p /tmp/blender && tar -xf /tmp/blender.tar.xz -C /tmp/blender --strip-components=1
!/tmp/blender/blender --version

In [ ]:
# 3. Render 10-Second 1080x1920 Cycles GPU Prototype (300 frames @ 16 samples + OPTIX GPU Denoiser -> ~10x faster!)
# Tip: Set SNELLIUS_DURATION_SEC=30 for a full 30s YouTube Short!
!SNELLIUS_DURATION_SEC=10 SNELLIUS_SAMPLES=16 /tmp/blender/blender --background --python src/build_snellius_scene.py

In [ ]:
# 4. Preview Frame 1 + Mux PNG Frames + Stereo Audio + Boxless Subtitles -> Final 1080x1920 MP4
from IPython.display import Image, display
display(Image('render_frames/frame_0001.png', width=320))
!mkdir -p output
!ffmpeg -y -framerate 30 -i render_frames/frame_%04d.png -i audio/snellius_test_audio.wav \
  -vf "ass=subtitles.ass" -c:v libx264 -profile:v high -crf 17 -pix_fmt yuv420p \
  -c:a aac -b:a 192k -shortest -movflags +faststart output/snellius_gpu_prototype.mp4 -loglevel warning
!ls -lh output/snellius_gpu_prototype.mp4

# Auto-download in Colab
try:
    from google.colab import files
    files.download('output/snellius_gpu_prototype.mp4')
except ImportError:
    print('Saved to output/snellius_gpu_prototype.mp4 — download from Kaggle Output panel!')